In [0]:
%run "../00_Setup_Config/project_config"

In [0]:
import dlt
from pyspark.sql.functions import col, when

def add_audit_metadata(df):
    return df.withColumn("ingestion_timestamp", current_timestamp()) \
             .withColumn("source_metadata", input_file_name())

@dlt.table(
    name="silver_layer.events_cleaned",
    comment="Silver Layer: Cleaned, casted, and deduplicated ecommerce events",
    table_properties={"quality": "silver"}
)
# Step 1: Data Quality Constraints (Expectations)
# Use 'expect_or_drop' to quarantine bad records without failing the pipeline
@dlt.expect_or_drop("valid_price", "price > 0")
@dlt.expect_or_drop("valid_user", "user_id IS NOT NULL")
@dlt.expect_or_drop("valid_event_time", "event_time IS NOT NULL")
def silver_transformation():
    return (
        # 1. Read from the Bronze table defined in your pipeline
        dlt.read_stream("events_raw")
        
        # 2. Type Casting & Selection: Do this early to reduce memory footprint
        # Casting to specific types (Long, Double) optimizes storage and shuffle performance
        .select(
            col("user_id").cast("long"),
            col("event_time").cast("timestamp"),
            col("price").cast("double"),
            "event_type", 
            "product_id",
            "category_code",
            "brand",
            "user_session"
        )
        
        # 3. Business Enrichment: Flag high-value transactions
        .withColumn("is_high_value", when(col("price") > 500, True).otherwise(False))
        
        # 4. Deduplication: Removes exact duplicates to ensure Gold aggregations are accurate
        # Always deduplicate AFTER filtering/casting for maximum efficiency
        .dropDuplicates(["user_id", "event_time", "product_id"])
    )

In [0]:
import dlt
from pyspark.sql.functions import col

@dlt.view(name="events_quarantine")
def events_quarantine():
    # Capture records that fail business rules (e.g., negative price or missing user)
    return dlt.read_stream("events_raw").filter(
        (col("price") <= 0) | (col("user_id").isNull())
    )